# Fahrzeugdatenauswertung

Die Fahrerkartendaten entstehen direkt im LKW: Der Fahrer steckt zu Beginn seiner Tätigkeit die Fahrerkarte ein und wählt jeweils manuell aus, ob er gerade fährt, arbeitet (z. B. Be- oder Entladen) oder eine Pause/Ruhezeit nimmt. Diese Informationen werden kontinuierlich auf der Karte gespeichert. Anschließend können die Fahrerkartendaten als Excel-Dateien heruntergeladen werden und enthalten eine chronologische Auflistung der Tätigkeiten der Fahrer: Lenkzeit, Arbeitszeit und Ruhezeit. Diese Tätigkeiten bilden gemeinsam die Einsatzzeit, die wiederum der Schichtzeit bzw. einer Tour eines Fahrers entspricht.

Das zentrale Problem besteht darin, dass aus den Rohdaten nicht automatisch erkennbar ist, wann eine Schicht beginnt und wann sie endet. Technisch beginnt die Schicht mit dem Einstecken der Fahrerkarte und endet mit dem Herausziehen, da anschließend automatisch Ruhezeit gebucht wird. Diese Logik ist jedoch in den Excel-Auszügen nicht direkt ablesbar, sodass Schichten nicht automatisch voneinander getrennt werden können. Bei größeren Datenmengen führt dies zu einem großen manuellen Aufwand.

Zusätzlich sollen **Pausen** innerhalb einer Schicht analysiert werden, insbesondere Pausen **über 30 Minuten**, **über 1 Stunde** und **über 2 Stunden**. Ruhezeiten treten jedoch sowohl am tatsächlichen Schichtende als auch als Pausen während der Schicht auf. Excel kann diese beiden Fälle nicht voneinander unterscheiden, und zusammenhängende Ruhezeiten, die sich über zwei Kalendertage erstrecken, werden ebenfalls nicht als Einheit erkannt. Ein Beispiel ist ein Fahrer, der spät am Abend beginnt und über Mitternacht hinweg eine Pause macht – diese wird im Export als zwei getrennte Ruhezeitblöcke angezeigt, obwohl es sich um eine einzige Pause handelt.

Um die Fahrerkartendaten sinnvoll auswerten zu können, wäre es daher notwendig **folgende Dinge zu ermitteln**:

* Schichtanfang und -ende zu erkennen,
* die Gesamtdauer einer Schicht zu berechnen,
* die einzelnen Tätigkeiten (Lenkzeit, Arbeitszeit, Ruhezeit) pro Schicht zu summieren,
* Pausen innerhalb der Schicht (>30 Min, >1 h, >2 h) zuverlässig zu identifizieren.

In [11]:
from pandas import read_excel

HEADERS = [
    "truck_status",
    "slot",
    "task_code",
    "task_description",
    "start_time",
    "end_time",
    "duration",
    "vehicle"
]

LOG_TRUCKER_STATUSES = ["Solo", "Manuelle Buchung", "Leer"]

In [16]:
# Load the data from Excel
data = read_excel("Rohdaten.xlsx", header=None)

# Set header row manually
data.columns = HEADERS

# Remove all intermediary header rows
data = data.loc[data.slot!="Steckplatz"]

# Create a new Date Column and fill it by taking ffill values
data.truck_status = data.truck_status.fillna("Leer")
data["date"] = data.truck_status
data["date"] = data["date"].map(lambda x: x if x not in LOG_TRUCKER_STATUSES else None)
data["date"] = data["date"].ffill()

# Create marker line to find standard booking lines
data["marker"] = data.truck_status.map(lambda x: True if x in LOG_TRUCKER_STATUSES else False)

In [19]:
# Look at all booking values
log_data = data.loc[data.marker == True]

In [20]:
log_data.head()

,truck_status,slot,task_code,task_description,start_time,end_time,duration,vehicle,date,marker
2,Manuelle Buchung,nicht erforderlich,r,Ruhezeit,02:00,04:46,02:46,NaN,31.03.2025,True
3,Solo,Fahrer,r,Ruhezeit,04:46,04:48,00:02,I 2696AX,31.03.2025,True
4,Solo,Fahrer,a,Arbeitszeit,04:48,04:52,00:04,I 2696AX,31.03.2025,True
5,Solo,Fahrer,l,Lenkzeit,04:52,06:57,02:05,I 2696AX,31.03.2025,True
6,Solo,Fahrer,a,Arbeitszeit,06:57,06:59,00:02,I 2696AX,31.03.2025,True
